In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [7]:
financials_df = pd.read_csv("../data/financials.csv")

print(financials_df.info())
print(financials_df.isna().sum())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 280 entries, 0 to 279
Data columns (total 17 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   symbol                          280 non-null    object 
 1   fiscalYear                      280 non-null    int64  
 2   reportedCurrency                280 non-null    object 
 3   period                          280 non-null    object 
 4   revenue                         280 non-null    int64  
 5   grossProfit                     280 non-null    int64  
 6   operatingIncome                 280 non-null    int64  
 7   netIncome                       280 non-null    int64  
 8   researchAndDevelopmentExpenses  280 non-null    int64  
 9   netInterestIncome               280 non-null    int64  
 10  totalDebt                       280 non-null    int64  
 11  totalStockholdersEquity         280 non-null    int64  
 12  capitalExpenditure              280 

In [11]:
financials_df.dtypes


symbol                             object
fiscalYear                          int64
reportedCurrency                   object
period                             object
revenue                             int64
grossProfit                         int64
operatingIncome                     int64
netIncome                           int64
researchAndDevelopmentExpenses      int64
netInterestIncome                   int64
totalDebt                           int64
totalStockholdersEquity             int64
capitalExpenditure                  int64
freeCashFlow                        int64
employeeCount                     float64
name                               object
sector                             object
dtype: object

In [14]:
int_cols = financials_df.select_dtypes(include='number').columns
(financials_df[int_cols] == 0).sum()

fiscalYear                          0
revenue                             0
grossProfit                         0
operatingIncome                     0
netIncome                           0
researchAndDevelopmentExpenses    125
netInterestIncome                  10
totalDebt                           1
totalStockholdersEquity             0
capitalExpenditure                 18
freeCashFlow                        0
employeeCount                       0
dtype: int64

In [15]:
pd.crosstab(financials_df["sector"], financials_df["capitalExpenditure"] == 0)

capitalExpenditure,False,True
sector,,
Consumer,50,0
Financials,42,18
Healthcare,40,0
Semiconductors,15,0
Tech,70,0
Transport,45,0


In [16]:
financials_df[financials_df["capitalExpenditure"] == 0][["symbol", "fiscalYear", "sector"]]

,symbol,fiscalYear,sector
25,BAC,2021,Financials
26,BAC,2022,Financials
27,BAC,2023,Financials
28,BAC,2024,Financials
29,BAC,2025,Financials
42,COIN,2023,Financials
43,COIN,2024,Financials
44,COIN,2025,Financials
120,JPM,2021,Financials
121,JPM,2022,Financials


In [26]:
financials_df[financials_df.symbol.isin(["JPM", "BAC", "C", "GS", "WFC"])][["symbol","revenue", "grossProfit"]]


,symbol,revenue,grossProfit
25,BAC,93851000000,93707000000
26,BAC,115053000000,92407000000
27,BAC,171912000000,94187000000
28,BAC,192434000000,96066000000
29,BAC,191567000000,107422000000
30,C,79868000000,75778000000
31,C,100220000000,69368000000
32,C,155382000000,67901000000
33,C,170707000000,71120000000
34,C,168302000000,74980000000


In [25]:
pd.crosstab(financials_df["sector"], financials_df["researchAndDevelopmentExpenses"] == 0)

researchAndDevelopmentExpenses,False,True
sector,,
Consumer,0,50
Financials,20,40
Healthcare,25,15
Semiconductors,15,0
Tech,70,0
Transport,25,20


## Data quality checks and cleaning decisions

The dataset has 280 rows (56 companies × 5 fiscal years, 2021–2025) and no missing values However, FMP reports unavailable figures as `0` rather than as null, so `isna()` does not reveal missing figures. Therefore, I checked for zero values in every numeric column and looked for patterns.

### 1. Bank "gross profit" is not a meaningful figure

Banks have no direct costs. Their income is the spread between interest earned and interest paid, and fees. Therefore, Gross profit in theory should undefined for them, but FMP provided a value despite it.

It is clear that the value is constructed rather than reported from above, where: JPMorgan's FY2021 gross profit (130.9bn) exceeds its revenue (127.2bn), which is impossible in normal circumstances.

The same issue also distorts revenue. Bank of America's revenue rises from 94bn (2021) to 192bn (2024), which is a growth of over 100%. These figures reveals increasing gross interest income rate instead of business growth, as interest expenses most likely rose in parallel.


**Decisions Made:**
- Exclude the Financials sector from all margin comparisons (Q2).
- Retain Financials in the revenue-growth analysis (Q1) but state this caveat explicitly in the interpretation.
- Where a bank-appropriate measure is needed, use `netInterestIncome`, which captures the money the bank actually earns.

### 2. Capital expenditure is zero for several banks

It was found that 18 rows reported a capex of exactly 0. All 18 of them are in Financials: BAC, JPM and WFC across all five years, plus COIN for three. No other sector has zeros in for capex.

Even though, Banks do actually purchase property and equipment, but they usually do not disclose it as a separate capital expenditure line. It is integrated into their other investing activities.

**Decision Made:** exclude these rows from the capital-intensity analysis (Q4). No imputation will be made, since any estimate or calculation would be invented.

### 3. R&D zeros are genuine, not missing

125 rows (45%) report zero R&D. The distribution is fully systematic:

| Sector | Reports R&D | Does not |
|---|---|---|
| Tech | 70 | 0 |
| Semiconductors | 15 | 0 |
| Healthcare | 25 | 15 |
| Transport | 25 | 20 |
| Financials | 20 | 40 |
| Consumer | 0 | 50 |

Every company in Tech and Semiconductors reports R&D, while Consumer companies does not. Within Transport, the split is between manufacturers like automakers, Boeing and operators like airlines, logistics. Moreover, in Healthcare, it's divided between drug developers and
insurers/providers. Lastly, in Financials, it's between fintechs and banks.

**Decision Made:** treat these zeros as real values. A company with no R&D spends 0% of revenue on R&D, which is a correct figure for the comparison.

### 4. Missing employee count

One row (UNH, FY2023) lacks an employee count, a null value. Thus, it will be excluded
from revenue-per-employee calculations only.

### 5. Fiscal-year misalignments (limitation)

In this investigation, fiscal years do not align across companies, Microsoft's ends in June, Nike's in May, while most others in December. "FY2023" therefore covers different twelve-month
periods depending on the company. This is accepted as a limitation rather than attempting to correct it, since this is usually a standard practice, but it will mean that year-on-year sector comparisons carry a few months of timing noise.